In [1]:
!pip install -q langchain langchain-groq gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.0 MB/s eta 0:00:00


In [ ]:
import os
import gradio as gr
from google.colab import userdata
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

# 1. Retrieve Groq API key securely from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")


def generate_titles(prompt_text):
  """Function to take a prompt and generate three title variations

  using different temperature settings via LangChain and Groq.
  """
  # Define a simple LangChain prompt template
  prompt_template = ChatPromptTemplate.from_messages([
      (
          "system",
          (
              "You are an expert copywriter. Generate a single catchy,"
              " compelling title for the given text. Return ONLY the title text"
              " without extra punctuation or quotes."
          ),
      ),
      ("human", "{input_text}"),
  ])

  output_parser = StrOutputParser()

  # 2. Instantiate ChatGroq models with 3 different temperature levels
  llm_low = ChatGroq(model="openai/gpt-oss-120b", temperature=0.1)
  llm_med = ChatGroq(model="openai/gpt-oss-120b", temperature=0.5)
  llm_high = ChatGroq(model="openai/gpt-oss-120b", temperature=0.9)

  # 3. Create simple LangChain LCEL chains
  chain_low = prompt_template | llm_low | output_parser
  chain_med = prompt_template | llm_med | output_parser
  chain_high = prompt_template | llm_high | output_parser

  # 4. Invoke chains with the user prompt
  title_low = chain_low.invoke({"input_text": prompt_text})
  title_med = chain_med.invoke({"input_text": prompt_text})
  title_high = chain_high.invoke({"input_text": prompt_text})

  return title_low.strip(), title_med.strip(), title_high.strip()


# 5. Build Gradio UI Wrapper
demo = gr.Interface(
    fn=generate_titles,
    inputs=gr.Textbox(
        lines=4,
        placeholder=(
            "Paste your article abstract, blog text, or description here..."
        ),
        label="Input Content / Prompt",
    ),
    outputs=[
        gr.Textbox(
            label=(
                "Low Temperature (0.1) — Focused, Direct & Deterministic"
            )
        ),
        gr.Textbox(label="Medium Temperature (0.5) — Balanced & Professional"),
        gr.Textbox(label="High Temperature (0.9) — Creative, Bold & Dynamic"),
    ],
    title="Multi-Temperature Title Generator",
    description=(
        "Powered by LangChain, Groq (Llama 3.3 70B), and Gradio. Compare how temperature changes AI creativity instantly."
    ),
)

# Launch the Gradio app (generates a public link in Colab)
demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5650f4a1124dd4d3fb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
